# 800 — CPTAC-3 external cohort: analysis-readiness assessment

**Purpose.** Validate the proposed 721-case source handoff from Notebook 113 and summarize the outstanding analytical conditions for evaluating the three frozen Phase 4 transcriptomic programs. The primary unit is one case with one primary RNA file; all seven supported lineages are reviewed separately.

**Run All validates source handoff and reconstructs the historical E exact-feature audit on 1,082 original STAR Count files when its new provenance-checked local cache is absent.** It does not contact GDC/PDC, download data, reselect source cases, project program scores or test biological replication. It does not modify the artifact registry or authorize biological inference.

Source cohort construction and quality provenance are owned by Notebooks 111–113. The scientific criteria and remaining G6–G8 design decisions are described in `docs/contracts/phase8/NOTEBOOK_800_ELIGIBILITY_AND_REPLICATION_CONTRACT_DRAFT.md`. The pre-existing local files under `data/interim/metadata` must already have been produced by the executed Notebook 113.

**Migration note (2026-11-02).** The full 1,082-file exact Phase 4 feature-presence scan originally executed inside 800 was not part of Notebook 113 G6's 721-file candidate audit. It is restored below without modifying the fixed upstream 721-case membership. The first run of this revised 800 is not yet execution-verified.


In [ ]:
# =============================================================================
# Imports, paths and exact-file utilities
# =============================================================================

import json
from pathlib import Path

import pandas as pd
from IPython.display import display

from pancancer_epigenetics.utils.file_checks import calculate_sha256
from pancancer_epigenetics.utils.paths import Paths

ROOT = Paths.root
META = Paths.metadata
REGISTRY_PATH = ROOT / "config" / "artifact_registry.json"
MANIFEST_PATH = META / "113_source_provenance_manifest.json"
READINESS_PATH = META / "113_cohort_readiness.json"

COHORT_NAME = "113_proposed_external_source_cohort.csv"
LEDGER_NAME = "113_primary_candidate_case_eligibility.csv"
LINEAGE_NAME = "113_proposed_source_cohort_lineage_counts.csv"
ATTRITION_NAME = "113_source_handoff_attrition.csv"
G6_NAME = "113_g6_raw_count_compatibility.csv"
G8_COVARIATES_NAME = "113_g8_covariate_coverage.csv"
G8_WORKFLOW_NAME = "113_g8_workflow_lineage_support.csv"
G8_CONTEXT_NAME = "113_g8_source_lineage_support.csv"

EXPECTED_LINEAGES = {
    "TCGA-HNSC": 36,
    "TCGA-LUAD": 178,
    "TCGA-LUSC": 89,
    "TCGA-PAAD": 82,
    "TCGA-SKCM": 138,
    "TCGA-STAD": 142,
    "TCGA-UCEC": 56,
}

def required_file(path):
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(
            f"Required local input missing: {path}. "
            "Use the existing executed Notebook 113 outputs; do not reconstruct the cohort in 800."
        )
    return path

def read_json(path):
    return json.loads(required_file(path).read_text(encoding="utf-8"))

def file_sha(path):
    return calculate_sha256(required_file(path))

def read_csv(name):
    return pd.read_csv(required_file(META / name), dtype="string")

def require_columns(frame, columns, label):
    absent = set(columns) - set(frame.columns)
    if absent:
        raise ValueError(f"{label}: missing columns {sorted(absent)}")

def check(condition, message):
    if not bool(condition):
        raise ValueError(message)

print("Repository:", ROOT)
print("Mode: offline source-handoff verification; program scoring disabled")


## 1. Accept the Notebook 113 source handoff

Validate the complete local provenance manifest, source input identities, every persisted 113 output, and the separate readiness record. This is an exact-content verification, not a rerun of Notebook 113. If a fingerprint disagrees, **stop** and reconcile the upstream files rather than selecting replacement cases.


In [ ]:
# =============================================================================
# Verify source manifest and readiness, including all recorded SHA-256 values
# =============================================================================

manifest = read_json(MANIFEST_PATH)
readiness = read_json(READINESS_PATH)
check(manifest.get("notebook") == 113, "Unexpected source manifest")
check(readiness.get("notebook") == 113, "Unexpected source readiness")
check(readiness.get("manifest_sha256") == file_sha(MANIFEST_PATH),
      "113 readiness does not match the source manifest")
check(manifest.get("cohort_frozen") is False, "113 manifest freeze state changed")
check(readiness.get("cohort_frozen") is False, "113 cohort is unexpectedly frozen")
check(readiness.get("external_analysis_authorized") is False,
      "Unexpected upstream biological-analysis authorization")

for relative, expected_sha in manifest["input_sha256"].items():
    check(file_sha(ROOT / relative) == expected_sha,
          f"113 source input fingerprint mismatch: {relative}")

for label, item in manifest.get("reviewed_evidence", {}).items():
    if isinstance(item, dict) and "path" in item and "sha256" in item:
        check(file_sha(ROOT / item["path"]) == item["sha256"],
              f"113 reviewed-source fingerprint mismatch: {label}")

output_info = manifest["outputs"]
for name, info in output_info.items():
    path = META / name
    check(file_sha(path) == info["sha256"],
          f"113 output fingerprint mismatch: {name}")

check(readiness.get("proposed_source_cohort_sha256") ==
      output_info[COHORT_NAME]["sha256"],
      "113 readiness/cohort manifest fingerprints disagree")

print("113 source manifest: VERIFIED")
print("113 source inputs: VERIFIED", len(manifest["input_sha256"]))
print("113 output files: VERIFIED", len(output_info))
print("113 readiness: source cohort proposed; external analysis not authorized")


## 2. Case-level membership and lineage reconciliation

The proposed source cohort is the already-computed G2 ∩ G3 ∩ G4 ∩ G6 intersection, **not** a new selection in Notebook 800. Reconcile it against the complete 1,082-case eligibility ledger and the independent per-lineage table. The 288 multiplicity cases remain outside the primary source universe; the 37 additional-ambiguous-file flags remain sensitivity metadata.


In [ ]:
# =============================================================================
# Validate 1,082-case ledger, 721-case handoff and exact inclusion intersection
# =============================================================================

cohort = read_csv(COHORT_NAME)
ledger = read_csv(LEDGER_NAME)
lineage_file = read_csv(LINEAGE_NAME)
attrition = read_csv(ATTRITION_NAME)

identity = ["case_uuid", "file_id", "sample_uuid", "aliquot_uuid"]
require_columns(cohort, identity + [
    "tcga_lineage", "g3_class", "g4_status", "g6_matrix_status",
    "source_handoff_status", "qc_status", "other_ambiguous_rna_flag",
], "113 cohort")
require_columns(ledger, identity + [
    "tcga_lineage", "g3_class", "g4_status", "g6_matrix_status",
    "source_handoff_status", "other_ambiguous_rna_flag",
], "113 complete case ledger")

check(len(ledger) == 1082 and ledger["case_uuid"].is_unique,
      "G2 1,082-case candidate ledger mismatch")
check(len(cohort) == 721, "Proposed source cohort is not 721 rows")
for label, frame in [("source cohort", cohort), ("G2 ledger", ledger)]:
    check(frame[identity].notna().all().all(), f"{label}: missing identity")
    check(frame["case_uuid"].is_unique and frame["file_id"].is_unique,
          f"{label}: case/file duplicates")

selected = (
    ledger["g3_class"].eq("COHORT_SUPPORTED_EXTERNAL")
    & ledger["g4_status"].eq("ACCEPTED")
    & ledger["g6_matrix_status"].eq("COMPATIBLE_RAW_COUNTS")
)
check(selected.notna().all(), "Incomplete upstream selection labels")
check(int(selected.sum()) == 721, "G3/G4/G6 intersection drift")
check(set(cohort["case_uuid"]) == set(ledger.loc[selected, "case_uuid"]),
      "113 cohort differs from the reviewed eligibility intersection")

included = ledger.loc[selected, ["case_uuid"] + identity[1:] + [
    "tcga_lineage", "g3_class", "g4_status", "g6_matrix_status"
]]
comparison = cohort[included.columns].merge(
    included, on="case_uuid", validate="one_to_one", suffixes=("_cohort", "_ledger")
)
check(len(comparison) == 721, "Membership case merge incomplete")
for column in included.columns.drop("case_uuid"):
    check(
        comparison[f"{column}_cohort"].eq(
            comparison[f"{column}_ledger"
        ).fillna(False).all(),
        f"113 cohort and ledger disagree in {column}",
    )
check(cohort["source_handoff_status"].eq("PROPOSED_SOURCE_COHORT").all(),
      "Unexpected 113 source membership status")
check(cohort["g3_class"].eq("COHORT_SUPPORTED_EXTERNAL").all(),
      "Cohort has unresolved G3 status")
check(cohort["g4_status"].eq("ACCEPTED").all(),
      "Cohort has unaccepted G4 lineage")
check(cohort["g6_matrix_status"].eq("COMPATIBLE_RAW_COUNTS").all(),
      "Cohort has incompatible raw counts")

check(int(ledger["other_ambiguous_rna_flag"].str.lower().eq("true").sum()) == 37,
      "G2 additional ambiguous RNA source flags changed")
check(int(manifest["source_counts"]["multiplicity_cases"]) == 288,
      "Upstream G2 multiplicity boundary changed")
check(int(manifest["source_counts"]["g2_simple_cases"]) == 1082,
      "Upstream G2 source candidate boundary changed")

print("Identity and exact source membership: VERIFIED")
print("G2 source candidates:", len(ledger), "| proposed cases:", len(cohort))
print("G3 residual donor-level independence uncertainty remains; QC is not certified")


In [ ]:
# =============================================================================
# Verify all seven source-to-TCGA lineage denominators and 113 attrition
# =============================================================================

require_columns(lineage_file, ["tcga_lineage", "n_cases"], "113 lineage counts")
require_columns(attrition, ["source_handoff_status", "n_cases"], "113 attrition")

observed = cohort.groupby("tcga_lineage")["case_uuid"].nunique().to_dict()
check(observed == EXPECTED_LINEAGES, f"Unexpected proposed lineage counts: {observed}")

reported = dict(zip(
    lineage_file["tcga_lineage"],
    pd.to_numeric(lineage_file["n_cases"], errors="raise").astype(int),
))
check(reported == EXPECTED_LINEAGES, "Source lineage summary differs from cohort")
check(pd.to_numeric(attrition["n_cases"], errors="raise").sum() == 1082,
      "113 case-attrition summary does not reconcile to G2")

lineage_support = (
    cohort.groupby("tcga_lineage", as_index=False)
    .agg(n_cases=("case_uuid", "nunique"),
         flagged_additional_ambiguous=(
             "other_ambiguous_rna_flag",
             lambda values: int(values.str.lower().eq("true").sum()),
         ))
    .sort_values("tcga_lineage")
    .reset_index(drop=True)
)
display(lineage_support)
print("G2 to proposed source cohort attrition:", len(ledger) - len(cohort))
print("All seven lineage denominators: VERIFIED; no lineage is yet inferentially approved")


## 3. G3 and G5: scope of valid claims

The source is externally supported at **cohort provenance level**. This does not certify zero shared individuals between CPTAC-3 and TCGA; undiscovered donor overlap remains possible. Read-level sequencing QC was not established across the whole cohort. These limitations remain visible rather than being converted into an additional donor-certification requirement or an invented QC pass criterion.


In [ ]:
# =============================================================================
# Report source provenance and QC without overclaiming donor/QC certification
# =============================================================================

require_columns(ledger, ["g3_class", "qc_status"], "113 ledger")
g3_counts = ledger["g3_class"].value_counts(dropna=False)
qc_counts = cohort["qc_status"].value_counts(dropna=False)

for label, expected in readiness.get("g3_class_counts", {}).items():
    check(int(g3_counts.get(label, 0)) == int(expected),
          f"G3 readiness count mismatch: {label}")
for label, expected in readiness.get("qc_status_counts", {}).items():
    check(int(ledger["qc_status"].eq(label).sum()) == int(expected),
          f"G5 readiness count mismatch: {label}")

display(pd.DataFrame({
    "g3_class": g3_counts.index.astype(str),
    "n_g2_cases": g3_counts.to_numpy(),
}))
display(qc_counts.rename_axis("cohort_qc_status").reset_index(name="n_cases"))
print("G3: cohort-level externality supported; individual donor overlap not genetically excluded")
print("G5: missing read-level QC remains QC_UNASSESSED, not QC_PASS")


## 4. G6: frozen reference identity and upstream count compatibility

Check that the frozen Phase 4 gene universe, program catalog and loading table have not changed. The 113 file-level compatibility assessment is consumed as an **audited 721-file source result**. A separate full-1,082-file STAR scan below restores the historical original E gene-coverage audit; these two scopes must never be conflated. Normalization and within-system projection parameters are not chosen or tuned in this run.


In [ ]:
# =============================================================================
# Verify frozen Phase 4 reference artifacts and 113 raw-count compatibility
# =============================================================================

registry = read_json(REGISTRY_PATH)
artifacts = registry["artifacts"]
reference_keys = {
    "shared_genes": "phase4.400.cross_system_shared_gene_universe",
    "program_catalog": "phase4.401.consensus_transcriptomic_program_catalog",
    "program_weights": "phase4.401.consensus_transcriptomic_gene_weights",
}

references = {}
for label, key in reference_keys.items():
    entry = artifacts[key]
    check(entry["status"] == "frozen", f"Phase 4 reference not frozen: {key}")
    target = ROOT / entry["path"]
    check(file_sha(target) == entry["sha256"],
          f"Frozen reference fingerprint mismatch: {key}")
    references[label] = pd.read_csv(target, dtype="string")
    check(len(references[label]) == entry["shape"][0],
          f"Frozen reference row-count mismatch: {key}")

gene_frame = references["shared_genes"]
require_columns(gene_frame, ["tumor_gene_id"], "Phase 4 gene universe")
gene_ids = gene_frame["tumor_gene_id"].str.replace(r"\.\d+$", "", regex=True)
check(len(gene_ids) == 2389 and gene_ids.notna().all() and gene_ids.is_unique,
      "Expected 2,389 unique frozen Ensembl features")
check(len(references["program_catalog"]) == 3, "Expected three frozen programs")
check(len(references["program_weights"]) == 3 * 2389,
      "Expected three complete 2,389-row loading vectors")

g6 = read_csv(G6_NAME)
require_columns(g6, [
    "case_uuid", "file_id", "status", "n_shared_genes_found",
    "n_duplicated_gene_ids", "has_unstranded",
], "113 G6 matrix audit")
check(g6["case_uuid"].is_unique and g6["file_id"].is_unique,
      "113 G6 matrix audit includes duplicated case/file identities")

g6_selected = g6.set_index("case_uuid").reindex(cohort["case_uuid"])
check(g6_selected["status"].eq("COMPATIBLE_RAW_COUNTS").all(),
      "One or more proposed files failed the 113 raw-count audit")
check(
    pd.to_numeric(g6_selected["n_shared_genes_found"], errors="raise").eq(2389).all(),
    "A proposed file lacks frozen genes",
)
check(
    pd.to_numeric(g6_selected["n_duplicated_gene_ids"], errors="raise").eq(0).all(),
    "A proposed file has duplicated version-normalized gene IDs",
)
check(g6_selected["has_unstranded"].str.lower().eq("true").all(),
      "A proposed file lacks the audited unstranded count field")
check(g6_selected["file_id"].to_numpy().tolist() == cohort["file_id"].tolist(),
      "G6 source file IDs differ from the proposed cohort")

print("Phase 4 frozen inputs: VERIFIED — three programs, 2,389 genes")
print("G6: 721 source raw-count files COMPATIBLE (113 evidence reused)")
print("Separate-cohort normalization and frozen-loading projection: NOT YET SPECIFIED")


## 4A. Original E — Frozen Phase 4 gene compatibility across all 1,082 source files

**Distinct population:** the full 1,082-case structural G2 universe from 111/112/113, **not** the proposed 721-case G3/G4/G6 intersection. Preserve the original E gene-identifier definitions and historical result comparisons: exact versioned Ensembl IDs, version-insensitive diagnostic, per-file detected frozen genes, and per-gene exact presence across all 1,082 RNA files.

This is an **input/feature-compatibility audit**, not read-level sequencing QC, program scoring or a new eligibility rule. It must not modify the frozen reference, alter the proposed 721 records, or promote the 288 cases with primary-sample multiplicity.

On first run, when the new E cache is absent, inspect the \`gene_id\` and \`unstranded\` columns of the 1,082 locally available STAR files and persist derived CSVs plus a source-fingerprint manifest in \`data/interim/metadata/\`. Later runs may reuse only those exact fingerprint-verified outputs. The presence of pre-refactor \`800_preflight_e_...\` files never substitutes for this code; those historical outputs are comparison evidence only.


In [ ]:
# =============================================================================
# Reconcile 1,082 exact source identities with the 112 technical audit
# =============================================================================
import hashlib

import numpy as np

TECH_112_PATH = META / "112_cptac3_rna_simple_primary_technical_inventory.csv"
HANDOFF_112_PATH = META / "112_cptac3_qc_provenance_handoff.json"
E_FILE_PATH = META / "800_cptac3_e_exact_feature_file_diagnostics.csv"
E_GENE_PATH = META / "800_cptac3_e_exact_frozen_gene_coverage.csv"
E_MANIFEST_PATH = META / "800_cptac3_e_exact_feature_manifest.json"
E_STAR_ROOT = ROOT / "data/raw/cptac3/rnaseq_star_counts"
E_STAR_SPECIAL = ("N_unmapped", "N_multimapping", "N_noFeature", "N_ambiguous")
E_AUDIT_SCHEMA = "phase8_E_full1082_exact_frozen_features_v1"
E_REBUILD_FULL_STAR_AUDIT = False  # Explicit opt-in to replace a valid old cache

source_112_manifest = read_json(HANDOFF_112_PATH)
receipt_112 = (source_112_manifest.get("outputs") or {}).get("technical_inventory")
check(isinstance(receipt_112, dict), "112 handoff lacks technical inventory provenance")
check(receipt_112.get("sha256") == file_sha(TECH_112_PATH),
      "112 STAR technical inventory SHA-256 disagrees with its handoff")
check(receipt_112.get("rows") == 1082, "112 technical handoff row count changed")
check(receipt_112.get("path") ==
      str(TECH_112_PATH.relative_to(ROOT)).replace("\\", "/"),
      "112 handoff path is not the expected technical inventory")
source_112 = pd.read_csv(required_file(TECH_112_PATH), dtype="string")
source_identity = ["case_uuid", "file_id", "sample_uuid", "aliquot_uuid"]
require_columns(source_112, source_identity + [
    "n_gene_rows", "n_duplicate_versionless_ids", "n_star_special_rows",
    "n_invalid_unstranded_gene_values",
    "unstranded_nonnegative_integer_vector", "unstranded_gene_count_total",
    "n_detected_star_genes",
    *["star_" + k.lower() for k in E_STAR_SPECIAL],
], "112 full STAR technical inventory")
require_columns(ledger, source_identity + [
    "filename", "size_bytes", "provider_md5", "primary_site",
    "disease_type", "source_handoff_status",
], "113 full G2 source ledger")
check(len(source_112) == len(ledger) == 1082,
      "E full-source audit requires 1,082 distinct G2 files")
check(source_112["case_uuid"].is_unique and source_112["file_id"].is_unique,
      "112 input is not one sample and file per case")

source_identity_left = ledger[source_identity].sort_values(
    source_identity, kind="stable").reset_index(drop=True)
source_identity_right = source_112[source_identity].sort_values(
    source_identity, kind="stable").reset_index(drop=True)
check(source_identity_left.equals(source_identity_right),
      "112 and 113 source case/file/sample/aliquot identities disagree")

e_frozen_exact = gene_frame["tumor_gene_id"].astype("string").reset_index(drop=True)
check(len(e_frozen_exact) == 2389
      and not e_frozen_exact.isna().any()
      and e_frozen_exact.is_unique, "Invalid original Phase 4 exact gene IDs")
e_frozen_bases = e_frozen_exact.str.replace(r"\.\d+$", "", regex=True)
check(e_frozen_bases.is_unique, "Frozen versionless Ensembl IDs collide")
e_frozen_set = set(e_frozen_exact.tolist())
e_frozen_base_set = set(e_frozen_bases.tolist())

e_source = ledger[source_identity + [
    "filename", "size_bytes", "provider_md5", "primary_site",
    "disease_type", "source_handoff_status",
]].sort_values(["case_uuid", "file_id"], kind="stable").reset_index(drop=True)

E_INPUT_FINGERPRINTS = {
    "112_technical_inventory_sha256": file_sha(TECH_112_PATH),
    "112_handoff_sha256": file_sha(HANDOFF_112_PATH),
    "113_source_manifest_sha256": file_sha(MANIFEST_PATH),
    "113_g2_ledger_sha256": file_sha(META / LEDGER_NAME),
    "phase4_shared_gene_reference_sha256": file_sha(
        ROOT / artifacts[reference_keys["shared_genes"]]["path"]
    ),
    "111_primary_source_index_sha256": manifest["input_sha256"][
        "data/interim/metadata/111_cptac3_provisional_primary_tumor_files.csv"
    ],
}
E_SOURCE_IDENTITY_SHA256 = hashlib.sha256(
    e_source[source_identity + ["filename", "size_bytes", "provider_md5"]]
    .to_csv(index=False, lineterminator="\n").encode("utf-8")
).hexdigest()
print("Original E sample scope:", len(e_source), "source files")
print("Phase 4 exact/versionless reference genes:", len(e_frozen_exact))
print("112 and 113 case/file/sample/aliquot identities: EXACT MATCH")
print("Proposed source cohort is unchanged:", len(cohort), "cases")


In [ ]:
# =============================================================================
# Regenerate original E exact, versionless and detection diagnostics on first run
# =============================================================================
e_cache_paths = [E_FILE_PATH, E_GENE_PATH, E_MANIFEST_PATH]
e_cache_exists = [path.is_file() for path in e_cache_paths]
if any(e_cache_exists) and not all(e_cache_exists) and not E_REBUILD_FULL_STAR_AUDIT:
    raise ValueError(
        "Partial E cache: never silently mix old and new feature evidence. "
        "Review files and explicitly set E_REBUILD_FULL_STAR_AUDIT=True to rebuild."
    )
E_SCAN_PERFORMED = E_REBUILD_FULL_STAR_AUDIT or not all(e_cache_exists)

if not E_SCAN_PERFORMED:
    prior_e_manifest = read_json(E_MANIFEST_PATH)
    check(prior_e_manifest.get("schema") == E_AUDIT_SCHEMA,
          "E source cache schema changed")
    check(prior_e_manifest.get("inputs") == E_INPUT_FINGERPRINTS,
          "E cache upstream source/frozen-feature SHA-256 drift")
    check(prior_e_manifest.get("source_identity_sha256") == E_SOURCE_IDENTITY_SHA256,
          "E cache exact G2 identities changed")
    check(prior_e_manifest.get("n_source_files") == 1082 and
          prior_e_manifest.get("n_frozen_genes") == 2389,
          "E source/frozen feature denominators changed")
    check(file_sha(E_FILE_PATH) == prior_e_manifest["file_output_sha256"]
          and file_sha(E_GENE_PATH) == prior_e_manifest["gene_output_sha256"],
          "E derived CSV changed since its source-verified execution")
    e_file_coverage = pd.read_csv(E_FILE_PATH, dtype="string")
    e_gene_coverage = pd.read_csv(E_GENE_PATH, dtype="string")
    print("Reused exact-input and SHA-256-verified full E coverage: 0 STAR files read")
else:
    e_records = []
    e_gene_presence = np.zeros(len(e_frozen_exact), dtype=np.int64)
    for i, row in enumerate(e_source.itertuples(index=False), 1):
        star_path = E_STAR_ROOT / str(row.file_id) / str(row.filename)
        if not star_path.is_file():
            raise FileNotFoundError(f"112-audited STAR payload unavailable: {star_path}")
        if star_path.stat().st_size != int(row.size_bytes):
            raise ValueError(f"GDC STAR source byte size disagrees for {row.file_id}")
        table = pd.read_csv(
            star_path, sep="\t", comment="#",
            usecols=["gene_id", "unstranded"],
            dtype={"gene_id": "string", "unstranded": "string"},
        )
        if table["gene_id"].isna().any():
            raise ValueError(f"Missing STAR gene IDs: {row.file_id}")
        special_mask = table["gene_id"].str.startswith("N_", na=False)
        special = table.loc[special_mask]
        genes = table.loc[~special_mask]
        g_ids = genes["gene_id"]
        if g_ids.empty or g_ids.duplicated().any():
            raise ValueError(f"Empty or duplicate exact STAR gene IDs: {row.file_id}")

        exact_presence = e_frozen_exact.isin(set(g_ids))
        e_gene_presence += exact_presence.to_numpy(dtype=np.int64)
        base_counts = g_ids.str.replace(r"\.\d+$", "", regex=True).value_counts(
            dropna=False
        )
        versionless_presence = e_frozen_bases.isin(
            set(g_ids.str.replace(r"\.\d+$", "", regex=True))
        )

        numeric = pd.to_numeric(genes["unstranded"], errors="coerce")
        counts = numeric.to_numpy(dtype="float64", na_value=np.nan)
        valid_counts = bool(
            np.isfinite(counts).all()
            and (counts >= 0).all()
            and (counts == np.floor(counts)).all()
        )
        special_numeric = pd.to_numeric(special["unstranded"], errors="coerce")
        n_special = {}
        for name in E_STAR_SPECIAL:
            values = special_numeric.loc[special["gene_id"].eq(name)]
            n_special["star_" + name.lower()] = (
                float(values.iloc[0]) if len(values) == 1
                and pd.notna(values.iloc[0]) else np.nan
            )

        e_records.append({
            "case_uuid": row.case_uuid, "file_id": row.file_id,
            "sample_uuid": row.sample_uuid, "aliquot_uuid": row.aliquot_uuid,
            "filename": row.filename, "size_bytes": row.size_bytes,
            "provider_md5": row.provider_md5,
            "primary_site": row.primary_site, "disease_type": row.disease_type,
            "n_gene_rows": int(len(g_ids)),
            "n_duplicate_versionless_ids": int(base_counts.gt(1).sum()),
            "n_star_special_rows": int(len(special)),
            "n_frozen_exact_present": int(exact_presence.sum()),
            "n_frozen_exact_missing": int(2389 - exact_presence.sum()),
            "n_frozen_versionless_diagnostic": int(versionless_presence.sum()),
            "n_frozen_versionless_only_diagnostic": int(
                (versionless_presence & ~exact_presence).sum()
            ),
            "unstranded_nonnegative_integer_vector": valid_counts,
            "n_invalid_unstranded_gene_values": int(
                (~np.isfinite(counts) | (counts < 0) |
                 (np.isfinite(counts) & (counts != np.floor(counts)))).sum()
            ),
            "unstranded_gene_count_total": (
                float(counts.sum()) if valid_counts else np.nan
            ),
            "n_detected_star_genes": (
                int((counts > 0).sum()) if valid_counts else pd.NA
            ),
            "n_detected_frozen_genes_exact": (
                int((counts[g_ids.isin(e_frozen_set).to_numpy()] > 0).sum())
                if valid_counts else pd.NA
            ),
            "selection_performed": False,
            "technical_qc_interpretation": (
                "EXACT_FEATURE_PRESENCE_NOT_SEQUENCING_QC"
            ),
            **n_special,
        })
        if i % 200 == 0 or i == len(e_source):
            print("Original E full STAR exact-feature scan:", i, "/", len(e_source))
    e_file_coverage = pd.DataFrame(e_records)
    e_gene_coverage = gene_frame[["gene_symbol", "tumor_gene_id"]].copy()
    e_gene_coverage["n_simple_files_exact_id_present"] = e_gene_presence
    e_gene_coverage["n_simple_files_inspected"] = len(e_file_coverage)
    e_gene_coverage["fraction_simple_files_exact_id_present"] = (
        e_gene_coverage["n_simple_files_exact_id_present"]
        / e_gene_coverage["n_simple_files_inspected"]
    )
    print("Original E source and gene coverage reconstructed; not yet persisted")

print("E feature audit:", len(e_file_coverage), "case/files ×",
      len(e_gene_coverage), "fixed genes")


In [ ]:
# =============================================================================
# Reconcile original E diagnostics with 112 and historical Phase 8 results
# =============================================================================
e_required = source_identity + [
    "filename", "size_bytes", "provider_md5", "n_gene_rows",
    "n_duplicate_versionless_ids", "n_star_special_rows",
    "n_frozen_exact_present", "n_frozen_exact_missing",
    "n_frozen_versionless_diagnostic",
    "n_frozen_versionless_only_diagnostic",
    "unstranded_nonnegative_integer_vector",
    "n_invalid_unstranded_gene_values", "unstranded_gene_count_total",
    "n_detected_star_genes", "n_detected_frozen_genes_exact",
    "selection_performed", *["star_" + k.lower() for k in E_STAR_SPECIAL],
]
require_columns(e_file_coverage, e_required, "full E per-source feature audit")
require_columns(e_gene_coverage, [
    "gene_symbol", "tumor_gene_id", "n_simple_files_exact_id_present",
    "n_simple_files_inspected", "fraction_simple_files_exact_id_present",
], "full E frozen-gene presence")
check(len(e_file_coverage) == 1082 and
      e_file_coverage["case_uuid"].is_unique and
      e_file_coverage["file_id"].is_unique,
      "Full 1082-file E assessment lost cases or duplicated identities")
check(len(e_gene_coverage) == 2389 and
      e_gene_coverage["tumor_gene_id"].tolist() == e_frozen_exact.tolist(),
      "Full E per-gene records differ from frozen Phase 4 feature order")

source_check = e_file_coverage[source_identity + [
    "filename", "size_bytes", "provider_md5"
]].sort_values(source_identity, kind="stable").reset_index(drop=True)
expected_source_check = e_source[source_identity + [
    "filename", "size_bytes", "provider_md5"
]].sort_values(source_identity, kind="stable").reset_index(drop=True)
for col in source_identity + ["filename", "provider_md5"]:
    check(source_check[col].astype("string").equals(
          expected_source_check[col].astype("string")),
          f"E cached file identity differs from 113 G2: {col}")
check(pd.to_numeric(source_check["size_bytes"], errors="raise").equals(
      pd.to_numeric(expected_source_check["size_bytes"], errors="raise")),
      "E source byte size differs from source index")

e_vs_112 = e_file_coverage.merge(
    source_112, on=source_identity, how="inner", validate="one_to_one",
    suffixes=("_e", "_112"),
)
check(len(e_vs_112) == 1082, "E full audit and 112 source diagnostics not aligned")
shared_numeric = [
    "n_gene_rows", "n_duplicate_versionless_ids",
    "n_star_special_rows", "n_invalid_unstranded_gene_values",
    "unstranded_gene_count_total", "n_detected_star_genes",
    *["star_" + k.lower() for k in E_STAR_SPECIAL],
]
for col in shared_numeric:
    left = pd.to_numeric(e_vs_112[col + "_e"], errors="raise")
    right = pd.to_numeric(e_vs_112[col + "_112"], errors="raise")
    check(left.equals(right), f"Full E recomputation disagrees with 112: {col}")
check(
    e_vs_112["unstranded_nonnegative_integer_vector_e"]
    .astype("string").str.lower().equals(
        e_vs_112["unstranded_nonnegative_integer_vector_112"]
        .astype("string").str.lower()
    ),
    "E STAR count-vector validity flags disagree with 112",
)
check(e_file_coverage["selection_performed"].astype("string").str.lower().eq(
    "false").all(), "E may not select files or set a QC threshold")

# Historical E baseline from 2026-10-23: 1,082 complete exact 2,389-gene
# feature sets, no versionless-only missing rescue, and valid count vectors.
# Disagreement is SOURCE/REFERENCE DRIFT to investigate, never a QC exclusion.
for col, expected in [
    ("n_frozen_exact_present", 2389),
    ("n_frozen_exact_missing", 0),
    ("n_frozen_versionless_only_diagnostic", 0),
    ("n_invalid_unstranded_gene_values", 0),
]:
    values = pd.to_numeric(e_file_coverage[col], errors="raise")
    check(values.eq(expected).all(),
          f"Historical E regression check failed: {col}")
gene_presence = pd.to_numeric(
    e_gene_coverage["n_simple_files_exact_id_present"], errors="raise"
)
check(gene_presence.eq(1082).all(),
      "Frozen gene exact-presence across 1,082 files changed")
check(pd.to_numeric(
    e_gene_coverage["n_simple_files_inspected"], errors="raise"
).eq(1082).all(), "Gene-level denominator changed")
check(pd.to_numeric(
    e_file_coverage["n_detected_frozen_genes_exact"], errors="raise"
).between(0, 2389).all(), "Invalid frozen-gene detection count")
check(
    int(pd.to_numeric(e_file_coverage["n_frozen_exact_present"]).sum())
    == int(gene_presence.sum()),
    "Sum of per-file exact gene presence does not match per-gene coverage"
)

e_full_context = (
    e_file_coverage.groupby(
        ["primary_site", "disease_type"], dropna=False
    ).agg(
        n_simple_source_files=("file_id", "nunique"),
        n_all_exact_frozen_genes=(
            "n_frozen_exact_missing",
            lambda x: int(pd.to_numeric(x).eq(0).sum())
        ),
        n_versionless_only_cases=(
            "n_frozen_versionless_only_diagnostic",
            lambda x: int(pd.to_numeric(x).gt(0).sum())
        ),
    ).reset_index()
)
check(e_full_context["n_simple_source_files"].sum() == 1082 and
      len(e_full_context) == 18, "E unharmonized source-context coverage drift")
check(set(cohort["file_id"]).issubset(set(e_file_coverage["file_id"])),
      "113 proposed cohort not contained in full E source universe")
display(e_file_coverage[[
    "n_gene_rows", "n_frozen_exact_present", "n_frozen_exact_missing",
    "n_frozen_versionless_only_diagnostic",
    "n_detected_frozen_genes_exact",
]].apply(pd.to_numeric, errors="raise").describe().T)
display(e_full_context.sort_values(
    "n_simple_source_files", ascending=False
).reset_index(drop=True))
print("Original E source universe: 1,082/1,082 exact 2,389-feature coverage")
print("Phase 8 SOURCE INPUT compatibility verified; no eligibility change")


In [ ]:
# =============================================================================
# Persist new derived E results and exact-input, integrity-verified cache receipt
# =============================================================================
if E_SCAN_PERFORMED:
    META.mkdir(parents=True, exist_ok=True)
    e_file_coverage.to_csv(E_FILE_PATH, index=False)
    e_gene_coverage.to_csv(E_GENE_PATH, index=False)
    e_manifest = {
        "notebook": 800,
        "schema": E_AUDIT_SCHEMA,
        "inputs": E_INPUT_FINGERPRINTS,
        "source_identity_sha256": E_SOURCE_IDENTITY_SHA256,
        "n_source_files": 1082,
        "n_frozen_genes": 2389,
        "file_output_sha256": file_sha(E_FILE_PATH),
        "gene_output_sha256": file_sha(E_GENE_PATH),
        "original_E_reference_comparison": {
            "all_1082_exact_2389_genes": True,
            "no_versionless_only_rescue_needed": True,
            "no_invalid_STAR_counts_detected": True,
        },
        "source_QC_certified": False,
        "case_selection_modified": False,
        "biological_replication_executed": False,
    }
    E_MANIFEST_PATH.write_text(
        json.dumps(e_manifest, indent=2, sort_keys=True) + "\n",
        encoding="utf-8",
    )
else:
    e_manifest = read_json(E_MANIFEST_PATH)

check(e_manifest["schema"] == E_AUDIT_SCHEMA and
      e_manifest["inputs"] == E_INPUT_FINGERPRINTS and
      e_manifest["source_identity_sha256"] == E_SOURCE_IDENTITY_SHA256,
      "E manifest no longer matches 112/113 and Phase 4 inputs")
check(file_sha(E_FILE_PATH) == e_manifest["file_output_sha256"] and
      file_sha(E_GENE_PATH) == e_manifest["gene_output_sha256"],
      "E derived artifacts changed after verification")
check(e_manifest.get("source_QC_certified") is False and
      e_manifest.get("biological_replication_executed") is False,
      "E source compatibility improperly promoted to QC or biology")
print("Full original E coverage cache:", E_MANIFEST_PATH.relative_to(ROOT))
print("E scanned raw STAR source payloads this run:", 1082 if E_SCAN_PERFORMED else 0)
print("Full E gene coverage reference and result hashes: VERIFIED")


## 5. G8: lineage support and confounder availability

Summarize the observed case counts, source workflow distribution and metadata coverage. **Not recorded** does not mean absent biologically. In particular, source-derived purity, proliferation, collection center, library preparation and sequencing batch must not be fabricated. A lineage can be marked non-evaluable later for lack of justified support or identifiability; this notebook does not apply arbitrary numerical exclusions.


In [ ]:
# =============================================================================
# Review 113 source covariates and actual proposed-cohort context support
# =============================================================================

g8_covariates = read_csv(G8_COVARIATES_NAME)
g8_workflows = read_csv(G8_WORKFLOW_NAME)
g8_contexts = read_csv(G8_CONTEXT_NAME)

require_columns(g8_covariates, [
    "covariate", "field_used", "n_present", "n_missing", "interpretation"
], "113 G8 covariates")
require_columns(g8_workflows, ["tcga_lineage", "workflow_type", "n_cases"],
                "113 G8 workflow support")
require_columns(g8_contexts, ["tcga_lineage", "g8_source_context", "n_cases"],
                "113 G8 source context support")
require_columns(ledger, ["workflow_type", "primary_site", "disease_type"],
                "113 case-level source context")

# The 113 G8 tables describe the complete G2 source candidates. For proposed-
# cohort support, summarize only the 721 exact accepted identities (without
# changing their membership or accessing expression outcomes).
selected_metadata = ledger.set_index("case_uuid").loc[
    cohort["case_uuid"].tolist()
].reset_index()
workflow_support = (
    selected_metadata.groupby(["tcga_lineage", "workflow_type"], dropna=False)
    .agg(n_cases=("case_uuid", "nunique"))
    .reset_index()
    .sort_values(["tcga_lineage", "workflow_type"])
)
context_support = (
    selected_metadata.groupby(
        ["tcga_lineage", "primary_site", "disease_type"], dropna=False
    )
    .agg(n_cases=("case_uuid", "nunique"))
    .reset_index()
    .sort_values(["tcga_lineage", "primary_site", "disease_type"])
)
check(int(workflow_support["n_cases"].sum()) == len(cohort),
      "Accepted-cohort workflow support does not reconcile")
check(int(context_support["n_cases"].sum()) == len(cohort),
      "Accepted-cohort context support does not reconcile")

print("G8 covariate field coverage among 1,082 G2 source candidates (not 721):")
display(g8_covariates)
print("Observed workflow distribution among 721 proposed source cases:")
display(workflow_support)
print("Observed source context distribution among 721 proposed source cases:")
display(context_support)
print("Adjustment feasibility / confounder identifiability: NOT ESTABLISHED")


## 6. G7 replication: analysis-design boundary

The candidate primary analysis is **within-lineage concordance of two disjoint frozen-program gene blocks**, tested against reference-matched negative controls. All three programs and all seven proposed lineages belong in the design review. A projected score alone, or correlations with its own constituent genes, cannot demonstrate structural replication.

The following choices still require a defensible, outcome-blind specification before biological scoring: gene-block partition (including reference-only balancing and repeated splits), CPTAC normalization and scale, primary concordance statistic, matched-control construction and draws, uncertainty estimator, multiplicity, minimum lineage support, feasible proliferation/purity sensitivities, and negative/insufficient-support labels.

These are **analytical decisions pending review**, not evidence that the source cohort is invalid. This run does not calculate program outcomes or inferential statistics.


In [ ]:
# =============================================================================
# Report the prespecified testing universe without running biological analysis
# =============================================================================

n_programs = len(references["program_catalog"])
n_lineages = len(EXPECTED_LINEAGES)
design_summary = pd.DataFrame([
    {"component": "Frozen transcriptomic programs", "value": n_programs,
     "decision": "FIXED"},
    {"component": "Proposed source lineages", "value": n_lineages,
     "decision": "SOURCE_MAPPING_AVAILABLE"},
    {"component": "Potential program-by-lineage comparisons",
     "value": n_programs * n_lineages, "decision": "EVALUABILITY_PENDING"},
    {"component": "G6 external normalization and projection",
     "value": "not run", "decision": "DESIGN_REQUIRED"},
    {"component": "G7 statistic, matched controls and multiplicity",
     "value": "not run", "decision": "DESIGN_REQUIRED"},
    {"component": "G8 confounder sensitivity and support",
     "value": "not run", "decision": "DESIGN_REQUIRED"},
])
display(design_summary)
print("No lineage or program was selected on the basis of CPTAC expression outcomes")


## 7. Persist a deterministic, non-inferential acceptance record

Write only concise local metadata records. The 113 files and artifact registry remain unchanged. The resulting verdict distinguishes a **verified source handoff** from **authorization to perform structural replication**.


In [ ]:
# =============================================================================
# Write local handoff acceptance and lineage support; never freeze the cohort
# =============================================================================

META.mkdir(parents=True, exist_ok=True)
acceptance_path = META / "800_cptac3_handoff_acceptance.json"
support_path = META / "800_cptac3_lineage_support.csv"

acceptance = {
    "notebook": 800,
    "source_notebook": 113,
    "source_manifest_sha256": file_sha(MANIFEST_PATH),
    "source_readiness_sha256": file_sha(READINESS_PATH),
    "source_cohort_sha256": output_info[COHORT_NAME]["sha256"],
    "frozen_shared_gene_universe_sha256": (
        artifacts[reference_keys["shared_genes"]]["sha256"]
    ),
    "frozen_program_catalog_sha256": (
        artifacts[reference_keys["program_catalog"]]["sha256"]
    ),
    "frozen_program_weights_sha256": (
        artifacts[reference_keys["program_weights"]]["sha256"]
    ),
    "n_source_candidate_cases": len(ledger),
    "n_proposed_source_cases": len(cohort),
    "n_proposed_source_files": int(cohort["file_id"].nunique()),
    "n_frozen_genes": len(gene_ids),
    "n_frozen_programs": n_programs,
    "full_E_source_files_audited": len(e_file_coverage),
    "full_E_frozen_features_audited": len(e_gene_coverage),
    "full_E_exact_coverage_status": "1082_OF_1082_ALL_2389_EXACT_GENES",
    "full_E_source_input_manifest_sha256": file_sha(E_MANIFEST_PATH),
    "full_E_file_coverage_sha256": file_sha(E_FILE_PATH),
    "full_E_per_gene_coverage_sha256": file_sha(E_GENE_PATH),
    "full_E_distinct_from_721_G6_scope": True,
    "lineage_counts": dict(sorted(observed.items())),
    "g3": "COHORT_SUPPORTED_EXTERNAL_WITH_RESIDUAL_DONOR_UNCERTAINTY",
    "g5": "SOURCE_INTEGRITY_VERIFIED_READ_QC_NOT_CERTIFIED",
    "g6": "SOURCE_RAW_COUNTS_COMPATIBLE_ANALYTIC_TRANSFORMATION_OPEN",
    "g7": "STRUCTURAL_REPLICATION_DESIGN_OPEN",
    "g8": "CONFOUNDER_IDENTIFIABILITY_AND_SUPPORT_POLICY_OPEN",
    "source_handoff_verified": True,
    "cohort_frozen": False,
    "external_analysis_authorized": False,
    "biological_replication_executed": False,
    "verdict": "HANDOFF_VERIFIED_ANALYSIS_DESIGN_PENDING",
}
lineage_support.to_csv(support_path, index=False)
acceptance_path.write_text(
    json.dumps(acceptance, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)

check(read_json(acceptance_path) == acceptance,
      "800 acceptance record did not persist correctly")
check(pd.read_csv(support_path).shape[0] == n_lineages,
      "800 lineage-support report did not persist correctly")

print("800 handoff acceptance:", acceptance["verdict"])
print("Acceptance record:", acceptance_path.relative_to(ROOT),
      file_sha(acceptance_path))
print("Lineage support:", support_path.relative_to(ROOT),
      file_sha(support_path))
print("No registry change, no program scoring, no external-inference authorization")


## Interpretation and next methodological step

**Accepted from Notebook 113:** the exact proposed 721-case/721-file source cohort and its seven accepted lineages, subject to passing the SHA-256 and identity checks above. G3 provides cohort-level externality with residual undocumented donor-overlap uncertainty. G5 does not certify full read-level sequencing QC.

**Recovered original E:** independently recomputed or SHA-256-reused 1,082 full-source STAR files × 2,389 exact Phase 4 features, preserving the historical E input-compatibility diagnostic. This does not turn 1,082 into the proposed analysis cohort, establish independent sequencing QC or authorize inference.

**Not yet decided:** G6 analytical normalization/projection, G7 complete structural test and matched null, and G8 inferential support/adjustment criteria. The 721 cases are **not automatically an inferentially frozen cohort**; biological conclusions, cross-cancer replication claims and drug-response interpretations remain outside the result of this notebook.

**Next step:** review and justify the remaining G6/G7/G8 choices, then implement a separately authorized biological analysis without changing the source handoff or using CPTAC outcomes to optimize eligibility.
